<a href="https://colab.research.google.com/github/KatherineElkins/programminghumanity2026/blob/main/pong_three_paradigms_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Choosing a Programming Paradigm

Programming paradigms are not merely three ways to rewrite the same program. Different problems make different structures useful.

1. **Procedural:** use a clear sequence of steps for a workflow.
2. **Functional:** transform inputs into outputs without changing the original data.
3. **Object-oriented:** organize a stateful system around interacting objects.

Each example includes a small change for you to make. Run the cells from top to bottom.


## 1. Procedural: processing an order

A checkout is naturally described as an ordered procedure: calculate the subtotal, apply a discount, calculate tax, and produce the total. The order matters, and each step updates the current result.


In [ ]:
def process_order(prices, member=False, tax_rate=0.0725):
    subtotal = sum(prices)
    print(f"Subtotal: ${subtotal:.2f}")

    discount = 0
    if member:
        discount = subtotal * 0.10
    discounted_subtotal = subtotal - discount
    print(f"Discount: -${discount:.2f}")

    tax = discounted_subtotal * tax_rate
    total = discounted_subtotal + tax
    print(f"Tax: ${tax:.2f}")
    print(f"Total: ${total:.2f}")
    return total


order = [18.00, 7.50, 4.25]
process_order(order, member=True)


### Modify the procedural version

Add a step: orders with a discounted subtotal of at least $25 receive **free shipping**; smaller orders add $5 shipping. Print the shipping charge and include it in the total.

**Questions:** Where must the new step occur? What would go wrong if shipping were added before the member discount?


In [ ]:
def process_order_with_shipping(prices, member=False, tax_rate=0.0725):
    subtotal = sum(prices)
    discount = subtotal * 0.10 if member else 0
    discounted_subtotal = subtotal - discount

    shipping = 0  # TODO: use an if statement to make this $0 or $5
    tax = discounted_subtotal * tax_rate
    total = discounted_subtotal + tax + shipping

    print(f"Subtotal: ${subtotal:.2f}")
    print(f"Discount: -${discount:.2f}")
    print(f"Shipping: ${shipping:.2f}")
    print(f"Tax: ${tax:.2f}")
    print(f"Total: ${total:.2f}")
    return total


process_order_with_shipping(order, member=True)


## 2. Functional: mathematical transformations

Functional programming grows out of mathematics: a function maps inputs to outputs. Pure functions do not change their inputs or rely on hidden state, so they can be composed into predictable transformation pipelines.

Here, one function derives percentage changes from a sequence of prices, another selects gains, and another calculates an average. Each result becomes the next function's input.


In [ ]:
def percent_changes(values):
    return tuple((current - previous) / previous * 100
                 for previous, current in zip(values, values[1:]))


def gains_only(changes):
    return tuple(change for change in changes if change > 0)


def average(values):
    return sum(values) / len(values) if values else 0


def average_gain(prices):
    return average(gains_only(percent_changes(prices)))


prices = (100, 103, 101, 106, 108, 104)
print("Daily percentage changes:", percent_changes(prices))
print("Average gain:", round(average_gain(prices), 2), "%")
print("Original prices are unchanged:", prices)


### Modify the functional version

Write `large_moves(changes, minimum_size)` so it returns a **new tuple** containing changes whose absolute value is at least `minimum_size`. Then use it to analyze the price sequence.

**Questions:** Why is `large_moves` a pure function? Why can it be tested without running the entire pipeline? How does this resemble composing mathematical functions?


In [ ]:
def large_moves(changes, minimum_size):
    return changes  # TODO: return changes where abs(change) >= minimum_size


changes = percent_changes(prices)
important_changes = large_moves(changes, minimum_size=3)
print("Moves of at least 3%:", important_changes)
print("Average size:", round(average(tuple(abs(x) for x in important_changes)), 2), "%")


## 3. Object-oriented: Pong

Pong is a stateful system of interacting entities. A `Paddle` and a `Ball` each preserve changing data and own relevant behavior; a `PongGame` coordinates them over time.

The simulation records the ball's path, so it runs reliably in Colab without opening a separate game window.


In [ ]:
import matplotlib.pyplot as plt


WIDTH, HEIGHT = 100, 60
LEFT_X, RIGHT_X = 6, 94
PADDLE_HALF = 8
BASE_SPEED = 1.6
STEPS = 180


def show_game(history, title):
    xs = [frame["ball_x"] for frame in history]
    ys = [frame["ball_y"] for frame in history]
    final = history[-1]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.set_facecolor("#10212b")
    ax.plot(xs, ys, color="#ffd166", linewidth=2)
    ax.scatter(xs[0], ys[0], color="#06d6a0", s=70, label="start")
    ax.scatter(xs[-1], ys[-1], color="#ef476f", s=70, label="finish")
    ax.vlines(LEFT_X, final["left_y"]-PADDLE_HALF, final["left_y"]+PADDLE_HALF, color="white", linewidth=7)
    ax.vlines(RIGHT_X, final["right_y"]-PADDLE_HALF, final["right_y"]+PADDLE_HALF, color="white", linewidth=7)
    ax.set(xlim=(0, WIDTH), ylim=(0, HEIGHT), title=title,
           xlabel="court width", ylabel="court height")
    ax.legend(loc="upper center", ncol=2)
    plt.show()
    print(f"Final score: {final['left_score']}–{final['right_score']}")


class Paddle:
    def __init__(self, x, y=30.0, speed=BASE_SPEED):
        self.x, self.y, self.speed = x, y, speed

    def move_toward(self, target_y):
        difference = target_y - self.y
        self.y += max(-self.speed, min(self.speed, difference))

    def can_hit(self, ball_y):
        return abs(ball_y - self.y) <= PADDLE_HALF


class Ball:
    def __init__(self):
        self.x, self.y = 50.0, 18.0
        self.vx, self.vy = 2.2, 1.35

    def move(self):
        self.x += self.vx
        self.y += self.vy

    def bounce_from_paddle(self, paddle_y):
        self.vx *= -1

    def bounce_vertical(self):
        self.vy *= -1

    def reset(self, direction):
        self.x, self.y = 50.0, 30.0
        self.vx, self.vy = 2.2 * direction, 1.35


class PongGame:
    def __init__(self, ball=None, paddle_speed=BASE_SPEED):
        self.left = Paddle(LEFT_X, speed=paddle_speed)
        self.right = Paddle(RIGHT_X, speed=paddle_speed)
        self.ball = ball if ball is not None else Ball()
        self.left_score = self.right_score = 0

    def snapshot(self):
        return {
            "ball_x": self.ball.x, "ball_y": self.ball.y,
            "left_y": self.left.y, "right_y": self.right.y,
            "left_score": self.left_score, "right_score": self.right_score,
        }

    def step(self):
        self.left.move_toward(self.ball.y)
        self.right.move_toward(self.ball.y)
        self.ball.move()

        if self.ball.y <= 0 or self.ball.y >= HEIGHT:
            self.ball.bounce_vertical()
            self.ball.y = max(0, min(HEIGHT, self.ball.y))

        hit_left = (self.ball.vx < 0 and self.ball.x <= LEFT_X
                    and self.left.can_hit(self.ball.y))
        hit_right = (self.ball.vx > 0 and self.ball.x >= RIGHT_X
                     and self.right.can_hit(self.ball.y))
        if hit_left:
            self.ball.bounce_from_paddle(self.left.y)
        elif hit_right:
            self.ball.bounce_from_paddle(self.right.y)

        if self.ball.x < 0:
            self.right_score += 1
            self.ball.reset(1)
        elif self.ball.x > WIDTH:
            self.left_score += 1
            self.ball.reset(-1)

    def run(self, steps=STEPS):
        history = [self.snapshot()]
        for _ in range(steps):
            self.step()
            history.append(self.snapshot())
        return history


oop_history = PongGame().run()
show_game(oop_history, "Object-Oriented Pong")


### Modify the object-oriented version

Add spin to the `Ball`. In `SpinBall.bounce_from_paddle`, calculate how far the ball strikes from the paddle's center and use that offset to change `self.vy`.

**Questions:** Why does the ball own its velocity? Why does Pong benefit from objects that preserve state over many time steps?


In [ ]:
class SpinBall(Ball):
    def bounce_from_paddle(self, paddle_y):
        super().bounce_from_paddle(paddle_y)
        impact_offset = (self.y - paddle_y) / PADDLE_HALF
        self.vy += 0  # TODO: add impact_offset * 0.8


spin_history = PongGame(ball=SpinBall()).run()
show_game(spin_history, "Object-Oriented Pong: add spin")


## When would you choose each paradigm?

| Paradigm | A good fit when… | This example |
|---|---|---|
| Procedural | the clearest structure is an ordered workflow | calculate an order one step at a time |
| Functional | mathematical relationships or data transformations are central | compose price transformations without mutating the data |
| Object-oriented | interacting entities preserve state and behavior over time | paddles and a ball participate in a game |

These are not absolute rules. A large program may combine paradigms. The goal is to choose structures that make the problem easier to understand and change.

**Discuss:**

1. Why would a full Pong game be awkward as the main example of functional programming?
2. Which modification was easiest to locate in the code, and why?
3. Where might a real program combine two or all three paradigms?
4. Does a paradigm merely change syntax, or does it change what a programmer notices about a problem?
